# RAG over ECHR + RIS + Swiss — retrieval → abstain → generate

Three corpora stay **separate on disk** (`data/echr_*.json`, `data/ris_*.json`,
`data/swiss_*.json`), combined **only in memory** into one tagged chunk list for a single
shared index. Every chunk carries `jurisdiction`, `source`, `lang`, `id`, and (ECHR)
`section`, so citations never cross jurisdictions.

- **ECHR** (EN) judgments are split by their standard headings; chunking **targets the Court's
  own assessment ("THE LAW")** and skips quoted statutes ("RELEVANT ... LAW") and
  separate/dissenting opinions. Judgments without those headings fall back to whole-text
  chunking (nothing dropped). Window ~600 words.
- **RIS** (DE) = Rechtssatz principles (one clean principle per chunk) **+ OGH civil-senate
  full decisions** (word-chunked, genre `decision`); criminal-senate records are excluded —
  the *Entfremdung* homonym (= misappropriation) — with a printed count.
- **Swiss** (DE) = entscheidsuche.ch full decisions (text from the ES `content` field),
  word-chunked, genre `decision`.
- Runs **end-to-end with no LLM installed** (retrieval-only). CPU-only.
- Dev default caps docs per source; set `MAX_DOCS_PER_SOURCE = None` for the full corpus.

## 1. Configuration

In [ ]:
from pathlib import Path

DATA_DIR  = Path("../data")
ECHR_FILE = DATA_DIR / "echr_parental_alienation.json"
RIS_FILE  = DATA_DIR / "ris_parental_alienation.json"
SWISS_FILE = DATA_DIR / "swiss_parental_alienation.json"
SOURCES = [("echr", ECHR_FILE), ("ris", RIS_FILE), ("swiss", SWISS_FILE)]   # source-pluggable

EMB_MODEL = "intfloat/multilingual-e5-base"   # needs query:/passage: prefixes
GEN_MODEL = "llama3.2"                         # local 3B; qwen2.5:0.5b is faster but garbles citations
USE_LLM   = True                              # False = force retrieval-only

# chunking — ECHR/Swiss/RIS decisions word-windowed; RIS principles one chunk each
CHUNK_SIZE_WORDS   = 600     # larger window for full judgments
CHUNK_OVERLAP      = 80
MAX_CHUNKS_PER_DOC = 60

# RIS: Rechtssatz principles are always indexed (one clean principle per chunk).
# RIS_INCLUDE_DECISIONS additionally indexes the 'Text' full-decision records, restricted to
# OGH *civil* senates (senate code 'Ob'): the criminal senates ('Os') hit the Entfremdung
# homonym (= misappropriation, §§ 133 ff StGB) and the AUSL/Bsw records are Austrian-institute
# summaries of ECtHR judgments — both off-topic for the family-law probe, so they are dropped
# with a printed count, never silently.
RIS_PRINCIPLES_ONLY  = False
RIS_INCLUDE_DECISIONS = True
_RIS_CIVIL_ID = "OGH0002"            # stable_id marker: real OGH decisions (not AUSL/Bsw)

# Swiss (entscheidsuche.ch): German-language full decisions; text lives in the ES-extracted
# `content` field (attachment.content), NOT `full_text`.
SWISS_INCLUDE = True

# ECHR sectioning: skip quoted statutes + dissents; index the Court's reasoning (+facts)
ECHR_SKIP_SECTIONS = {"RELEVANT_LAW", "OPINION"}
ECHR_LAW_ONLY      = False   # True = index ONLY the "THE LAW" assessment section

# display snippet (sentence-aligned)
SNIPPET_TARGET = 2200
SNIPPET_MAX    = 2500

MAX_DOCS_PER_SOURCE = None    # dev cap; None = full corpus
TOP_K = 6
# SCORE_FLOOR = 0.80         # optional Type-2 retrieval floor (see answer())

# --- genre-aware retrieval + MMR diversity (ECHR ingestion-discipline fix) ---
# ECHR communicated cases (pending applications, no decided ruling) carry a clean
# "SUBJECT MATTER OF THE CASE" restatement that paraphrases the query and out-scores real
# merits judgments, then floods the top-k as near-duplicates. We keep them in the corpus
# (correct data for "is this topic litigated / how often") but ROUTE them out of
# law-questions. Genre is metadata derived from existing text — no re-scrape, no re-embed.
LAW_GENRE_EXCLUDE = {"communicated"}   # genres dropped for "what is the law" questions
FETCH_K           = 30                 # wide candidate pool fetched before MMR + filtering
MMR_LAMBDA        = 0.7                # relevance vs diversity (1.0 = pure relevance)
MMR_DUP_CEIL      = 0.95               # cruder fallback: hard-drop near-duplicate candidates

EMB_CACHE = DATA_DIR / "rag_echr_ris_emb_cache.npz"

print("inputs:", [str(p.name) for _, p in SOURCES])
print(f"chunk={CHUNK_SIZE_WORDS}w/{CHUNK_OVERLAP}o | ECHR skip={ECHR_SKIP_SECTIONS} law_only={ECHR_LAW_ONLY} | dev_cap={MAX_DOCS_PER_SOURCE}")
print(f"RIS decisions={RIS_INCLUDE_DECISIONS} (civil only) | Swiss={SWISS_INCLUDE}")
print(f"genre routing: exclude={LAW_GENRE_EXCLUDE} | MMR fetch_k={FETCH_K} lambda={MMR_LAMBDA}")

inputs: ['echr_parental_alienation.json', 'ris_parental_alienation.json', 'swiss_parental_alienation.json']
chunk=600w/80o | ECHR skip={'OPINION', 'RELEVANT_LAW'} law_only=False | dev_cap=None
RIS decisions=True (civil only) | Swiss=True
genre routing: exclude={'communicated'} | MMR fetch_k=30 lambda=0.7


## 2. Load + normalise (each source independently)
ECHR `date` cascade: `judgementdate` (timestamp) → **`ecli`** (`ECLI:CE:ECHR:YYYY:MMDD`,
verified to match judgementdate) → a date phrase in `full_text` → "" — all rendered
**YYYY-MM-DD**. `conclusion` is metadata only (Article-8 finding, never the custody
outcome). RIS keys are lowercase; principle = text between the `Rechtssatz` and
`Entscheidungstexte` labels; applied decisions kept as `applied_decisions` metadata.

In [ ]:
import json
import re
from datetime import datetime

_EN_MONTHS = {m: i for i, m in enumerate(
    ["january", "february", "march", "april", "may", "june", "july", "august",
     "september", "october", "november", "december"], 1)}
_FT_DATE = re.compile(
    r"(?:communicated on|published on|strasbourg,?|judgment\s+strasbourg|decision\s+strasbourg)"
    r"\s+(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", re.IGNORECASE)


def load_json_records(path):
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):     # RIS single-result-as-dict guard
        data = [data]
    return data


def _echr_date(r):
    # 1) judgementdate timestamp e.g. "19/01/2016 00:00:00"
    raw = (r.get("judgementdate") or "").strip()
    if raw:
        try:
            return datetime.strptime(raw.split()[0], "%d/%m/%Y").date().isoformat()
        except (ValueError, IndexError):
            pass
    # 2) ECLI encodes the date: ECLI:CE:ECHR:YYYY:MMDD...
    m = re.match(r"ECLI:CE:ECHR:(\d{4}):(\d{2})(\d{2})", r.get("ecli", "") or "")
    if m:
        y, mo, d = m.groups()
        if 1 <= int(mo) <= 12 and 1 <= int(d) <= 31:
            return f"{y}-{mo}-{d}"
    # 3) a date phrase in the body ("Communicated on 24 August 2015", "STRASBOURG 19 January 2016")
    fm = _FT_DATE.search((r.get("full_text", "") or "")[:4000])
    if fm:
        d, mon, y = fm.groups()
        mi = _EN_MONTHS.get(mon.lower())
        if mi:
            return f"{y}-{mi:02d}-{int(d):02d}"
    return ""


def split_ris_principle(full_text):
    lines = (full_text or "").split("\n")

    def find(label):
        for i, l in enumerate(lines):
            if l.strip() == label:
                return i
        return -1

    i_rs = find("Rechtssatz")
    if i_rs == -1:
        return "", ""
    i_et = find("Entscheidungstexte")
    i_ecli = find("European Case Law Identifier")
    end = i_et if i_et != -1 else (i_ecli if i_ecli != -1 else len(lines))
    principle = "\n".join(lines[i_rs + 1:end]).strip()
    applied = ""
    if i_et != -1:
        a_end = i_ecli if i_ecli != -1 else len(lines)
        applied = "\n".join(lines[i_et + 1:a_end]).strip()
    return principle, applied


def normalise_echr(r):
    iid = r.get("itemid", "") or ""
    return {
        "id": iid,
        "title": r.get("docname", "") or iid,
        "text": r.get("full_text", "") or "",
        "jurisdiction": "ECHR",
        "source": "echr",
        "country": r.get("respondent", "") or "",
        "date": _echr_date(r),
        "lang": "en",
        "url": f"https://hudoc.echr.coe.int/eng?i={iid}" if iid else "",
        "matched_keywords": r.get("matched_keywords") or [],
        "applied_decisions": [],
        "genre_hint": None,                                    # ECHR genre from doctype/markers
        "meta": {"conclusion": r.get("conclusion", ""), "article": r.get("article", ""),
                 "importance": r.get("importance", ""), "appno": r.get("appno", ""),
                 "doctype": r.get("doctype", ""),            # HEJUD/HEDEC/HECOM -> genre
                 "doctypebranch": r.get("doctypebranch", "")},
    }


def normalise_ris(r):
    """RIS Rechtssatz -> genre 'principle' (the distilled principle text only);
    RIS 'Text' full decision -> genre 'decision' (whole decision text, word-chunked)."""
    rid = r.get("id", "") or ""
    gericht = r.get("gericht", "") or "OGH"
    rsnum = (r.get("rechtssatznummern") or "").strip()
    gz = (r.get("geschaeftszahl") or "").split(";")[0].strip()
    is_principle = r.get("dokumenttyp") == "Rechtssatz"
    if is_principle:
        text, _ = split_ris_principle(r.get("full_text", ""))
        title = (f"{gericht} {rsnum}" if rsnum else f"{gericht} {gz}").strip()
    else:
        text = r.get("full_text", "") or ""
        title = f"{gericht} {gz}".strip()
    return {
        "id": rid,
        "title": title or rid,
        "text": text,
        "jurisdiction": "AT (OGH)",
        "source": "ris",
        "country": "AT",
        "date": (r.get("entscheidungsdatum") or "")[:10],
        "lang": "de",
        "url": r.get("content_url_html", "") or r.get("source_url", ""),
        "matched_keywords": r.get("matched_keywords") or [],
        "applied_decisions": r.get("entscheidungstexte") or [],
        "genre_hint": "principle" if is_principle else "decision",
        "meta": {"geschaeftszahl": r.get("geschaeftszahl", ""), "rechtssatznummern": rsnum,
                 "rechtsgebiete": r.get("rechtsgebiete", ""),
                 "dokumenttyp": r.get("dokumenttyp", "")},
    }


def normalise_swiss(r):
    """entscheidsuche.ch record: full decision text is in `content` (ES attachment.content)."""
    rid = r.get("stable_id", "") or r.get("Signatur", "") or ""
    return {
        "id": rid,
        "title": (r.get("title") or rid).strip(),
        "text": r.get("content", "") or "",
        "jurisdiction": "CH",
        "source": "swiss",
        "country": "CH",
        "date": (r.get("Datum") or "")[:10],
        "lang": r.get("lang", "de") or "de",
        "url": r.get("source_url", "") or r.get("content_url", ""),
        "matched_keywords": r.get("matched_keywords") or [],
        "applied_decisions": [],
        "genre_hint": "decision",
        "meta": {"canton": r.get("canton", ""), "court_type": r.get("court_type", ""),
                 "hierarchy": r.get("hierarchy", ""), "reference": r.get("reference", ""),
                 "collection": r.get("collection", "")},
    }


NORMALISERS = {"echr": normalise_echr, "ris": normalise_ris, "swiss": normalise_swiss}
print("normalisers ready:", list(NORMALISERS))

normalisers ready: ['echr', 'ris', 'swiss']


## 3. ECHR section splitter + sentence snippet + chunking
ECHR headings are uppercase and glued to the text (`THE LAWI. ALLEGED VIOLATION…`), so they
are matched by regex. A judgment with a `THE LAW` heading is sectioned; otherwise it falls
back to whole-text. Display snippets are cut on a sentence boundary (~2000–2500 chars, never
mid-word); the full chunk text is preserved for the LLM context.

In [ ]:
# uppercase, glued, case-sensitive anchors
ECHR_ANCHORS = [
    (re.compile(r"PROCEDURE(?=[0-9IVX])"), "PROCEDURE"),
    (re.compile(r"THE FACTS(?=[0-9IVX]|\s)"), "FACTS"),
    (re.compile(r"THE CIRCUMSTANCES OF THE CASE"), "FACTS"),
    (re.compile(r"RELEVANT (?:DOMESTIC|LEGAL|INTERNATIONAL|EUROPEAN|COMPARATIVE|COUNCIL)"
                r"[A-Z ]{0,40}?(?:LAW|FRAMEWORK|MATERIAL|PRACTICE|TEXT)"), "RELEVANT_LAW"),
    (re.compile(r"(?:AS TO )?THE LAW(?![a-z])|THE COURT[^A-Za-z]{0,2}S ASSESSMENT"), "LAW"),
    (re.compile(r"FOR THESE REASONS"), "OPERATIVE"),
    (re.compile(r"(?:JOINT |PARTLY )?(?:DISSENTING|SEPARATE|CONCURRING) OPINION"), "OPINION"),
]


def echr_sections(full_text):
    # returns list of (label, text) or None if no 'THE LAW' heading is found
    marks = sorted((m.start(), lab) for rx, lab in ECHR_ANCHORS for m in rx.finditer(full_text))
    if not marks or not any(lab == "LAW" for _, lab in marks):
        return None
    segs = []
    if marks[0][0] > 0:
        segs.append(("HEADER", full_text[:marks[0][0]]))
    for i, (pos, lab) in enumerate(marks):
        end = marks[i + 1][0] if i + 1 < len(marks) else len(full_text)
        segs.append((lab, full_text[pos:end]))
    return segs


def _echr_keep(section):
    if section == "unparsed":
        return True                      # fallback: keep everything
    if ECHR_LAW_ONLY:
        return section == "LAW"
    return section not in ECHR_SKIP_SECTIONS


_WS = re.compile(r"\S+")
_SENT = re.compile(r"(?<=[.!?])\s+")


def chunk_words(text, size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP, max_chunks=MAX_CHUNKS_PER_DOC):
    words = _WS.findall(text or "")
    if not words:
        return []
    if len(words) <= size:
        return [" ".join(words)]
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        out.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return out[:max_chunks] if max_chunks else out


def sentence_snippet(text, target=SNIPPET_TARGET, hard_max=SNIPPET_MAX):
    text = (text or "").strip()
    if len(text) <= target:
        return text
    out = ""
    for s in _SENT.split(text):
        if not out:
            out = s
        elif len(out) + 1 + len(s) <= hard_max:
            out = out + " " + s
        else:
            break
        if len(out) >= target:
            break
    if len(out) > hard_max:                 # single huge sentence -> cut on a word boundary
        cut = out[:hard_max]
        out = cut[:cut.rfind(" ")] if " " in cut else cut
    return out + ("…" if len(out) < len(text) else "")


def build_chunks(records):
    """ECHR -> section-aware pieces; RIS principle -> one clean chunk;
    RIS decision / Swiss decision -> word-windowed whole text (no ECHR-style sections)."""
    chunks = []
    for rec in records:
        genre = assign_genre(rec)                              # first-class chunk field
        if rec["source"] == "echr":
            secs = echr_sections(rec["text"])
            if secs is None:
                pieces = [("unparsed", rec["text"])]            # safe fallback
            else:
                pieces = [(lab, seg) for lab, seg in secs if _echr_keep(lab)]
        elif genre == "principle":
            pieces = [("principle", rec["text"])]
        else:
            pieces = [("decision", rec["text"])]
        idx = 0
        for section, segtext in pieces:
            subs = [segtext] if section == "principle" else chunk_words(segtext)
            for piece in subs:
                if not piece.strip():
                    continue
                chunks.append({
                    "chunk_id": f"{rec['source']}:{rec['id']}:{idx}",
                    "doc_id": rec["id"], "source": rec["source"],
                    "jurisdiction": rec["jurisdiction"], "section": section,
                    "genre": genre,
                    "title": rec["title"], "url": rec["url"], "date": rec["date"],
                    "lang": rec["lang"], "country": rec["country"],
                    "matched_keywords": rec["matched_keywords"],
                    "applied_decisions": rec.get("applied_decisions", []),
                    "text": piece,
                })
                idx += 1
    return chunks


print("section splitter + chunker ready")

section splitter + chunker ready


## 3b. Genre — a first-class chunk field (reusable helper)
ECHR documents come in three **genres** that differ in answer-value for a *"what is the law"*
question:

- **`merits`** — decided judgments (`doctype=HEJUD`). The Court states the principle. **Wanted.**
- **`admissibility`** — admissibility decisions (`HEDEC`). Often principle-bearing. **Kept.**
- **`communicated`** — *pending* applications (`HECOM`): a clean `SUBJECT MATTER OF THE CASE`
  restatement + `QUESTIONS TO THE PARTIES`, **no decided ruling**. High query-similarity,
  zero answer-value, near-duplicate templates that flood the top-k. **Routed out of law-questions
  (not deleted)** — they are the *correct* data for "is this topic being litigated / how often".

Genre is derived from existing fields/text (no re-scrape, no re-embed): primary signal is the
HUDOC `doctype`, with a textual-marker fallback (`Communicated on`, `SUBJECT MATTER OF THE
CASE`, `QUESTIONS TO THE PARTIES`) for records lacking the field. RIS carries **no** ECHR
genre — its Rechtssatz principles are tagged `principle` (this fix is ECHR-only). The next
cell cross-checks the marker detector against the existing *unparsed* parser-flag before the
genre is wired into retrieval.

In [ ]:
# --- reusable ECHR genre helper (dependency-free: needs only `re`) ---
# Copy-paste-able into the diachronic / framing notebooks; classifies an ECHR document into
# {merits, admissibility, communicated} from its HUDOC doctype and/or its body text.

ECHR_GENRES = ("merits", "admissibility", "communicated", "other")
LOW_INFO_GENRES = {"communicated"}        # genres that carry no decided principle

_DOCTYPE_GENRE = {"HEJUD": "merits", "HEDEC": "admissibility", "HECOM": "communicated"}
_GENRE_LAW_RE = re.compile(r"(?:AS TO )?THE LAW(?![a-z])|THE COURT[^A-Za-z]{0,2}S ASSESSMENT")
_RE_COMMUNICATED_ON = re.compile(r"Communicated on")
_MARK_SUBJECT = "SUBJECT MATTER OF THE CASE"
_MARK_QUESTIONS = "QUESTIONS TO THE PARTIES"


def echr_genre_from_markers(text):
    """Genre from body-text markers only (no doctype) — used for the cross-check."""
    t = text or ""
    if _RE_COMMUNICATED_ON.search(t) or _MARK_QUESTIONS in t:
        return "communicated"
    if _MARK_SUBJECT in t and not _GENRE_LAW_RE.search(t):
        return "communicated"            # subject-matter restatement with no 'THE LAW' = pending
    if _GENRE_LAW_RE.search(t):
        return "merits"                  # has the Court's assessment; HEDEC refined via doctype
    return "other"


def echr_genre(text, doctype=""):
    """Primary classifier: trust the HUDOC doctype, fall back to body markers."""
    g = _DOCTYPE_GENRE.get((doctype or "").upper())
    return g if g else echr_genre_from_markers(text)


def assign_genre(rec):
    """Genre for any corpus record. ECHR -> merits/admissibility/communicated;
    non-ECHR sources carry a genre_hint from their normaliser:
    'principle' (RIS Rechtssatz) or 'decision' (RIS Text / Swiss full decision)."""
    if rec.get("source") != "echr":
        return rec.get("genre_hint") or "principle"
    return echr_genre(rec.get("text", ""), rec.get("meta", {}).get("doctype", ""))


print("genre helper ready:", ECHR_GENRES, "| low-info:", LOW_INFO_GENRES)

genre helper ready: ('merits', 'admissibility', 'communicated', 'other') | low-info: {'communicated'}


## 4. Build the in-memory corpus + report (dates, sections)

In [ ]:
from collections import Counter

records, chunks = [], []
DATA_PRESENT = all(p.exists() for _, p in SOURCES)

_RIS_CRIMINAL = re.compile(r"\d{1,3}\s*Os\b", re.IGNORECASE)   # OGH criminal senate (Entfremdung homonym)

if DATA_PRESENT:
    for src, path in SOURCES:
        raw = load_json_records(path) or []
        if src == "ris":
            n_all = len(raw)
            principles = [r for r in raw if r.get("dokumenttyp") == "Rechtssatz"]
            if RIS_PRINCIPLES_ONLY or not RIS_INCLUDE_DECISIONS:
                raw = principles
                print(f"  ris : {len(raw)} principles (dropped {n_all - len(raw)} 'Text' decisions)")
            else:
                texts = [r for r in raw if r.get("dokumenttyp") == "Text"]
                civil = [r for r in texts
                         if _RIS_CIVIL_ID in (r.get("id") or "")
                         and not _RIS_CRIMINAL.search((r.get("geschaeftszahl") or "").split(";")[0])]
                print(f"  ris : {len(principles)} principles + {len(civil)} civil decisions "
                      f"(dropped {len(texts) - len(civil)} criminal-senate/AUSL 'Text' records "
                      f"— Entfremdung homonym / ECtHR summaries)")
                raw = principles + civil
        if src == "swiss":
            if not SWISS_INCLUDE:
                print("  swiss: skipped (SWISS_INCLUDE=False)")
                continue
            n_all = len(raw)
            raw = [r for r in raw
                   if "rechenschaftsbericht" not in str(r.get("title", "")).lower()]
            if n_all - len(raw):
                print(f"  swiss: dropped {n_all - len(raw)} Rechenschaftsbericht records "
                      f"(court annual reports, not case law — multi-case digests that "
                      f"flood the top-k)")
        if MAX_DOCS_PER_SOURCE:
            raw = raw[:MAX_DOCS_PER_SOURCE]
        recs = [NORMALISERS[src](r) for r in raw]
        n_empty = sum(1 for r in recs if not (r["text"] or "").strip())
        if n_empty:
            print(f"  !! {src}: {n_empty} records with empty text (kept out of the index)")
            recs = [r for r in recs if (r["text"] or "").strip()]
        records += recs
        print(f"  {src:5s}: {len(recs)} records from {path.name}")

    # reports the task asks for
    echr_recs = [r for r in records if r["source"] == "echr"]
    if echr_recs:
        dated = sum(1 for r in echr_recs if r["date"])
        parsed = sum(1 for r in echr_recs if echr_sections(r["text"]) is not None)
        print(f"\nECHR dates: {dated}/{len(echr_recs)} have YYYY-MM-DD, {len(echr_recs) - dated} n.d.")
        print(f"ECHR sections: {parsed}/{len(echr_recs)} parsed; {len(echr_recs) - parsed} fell back to whole-text")

    chunks = build_chunks(records)
    print(f"\nrecords: {len(records)}  ->  chunks: {len(chunks)}")
    print("chunks by jurisdiction:", dict(Counter(c["jurisdiction"] for c in chunks)))
    print("chunks by language    :", dict(Counter(c["lang"] for c in chunks)))
    print("ECHR chunks by section :", dict(Counter(c["section"] for c in chunks if c["source"] == "echr")))
else:
    for _, p in SOURCES:
        if not p.exists():
            print("missing:", p)

  echr : 1116 records from echr_parental_alienation.json
  ris : 38 principles + 479 civil decisions (dropped 31 criminal-senate/AUSL 'Text' records — Entfremdung homonym / ECtHR summaries)
  ris  : 517 records from ris_parental_alienation.json
  swiss: dropped 24 Rechenschaftsbericht records (court annual reports, not case law — multi-case digests that flood the top-k)
  swiss: 2007 records from swiss_parental_alienation.json

ECHR dates: 1114/1116 have YYYY-MM-DD, 2 n.d.
ECHR sections: 873/1116 parsed; 243 fell back to whole-text

records: 3640  ->  chunks: 42055
chunks by jurisdiction: {'ECHR': 15136, 'AT (OGH)': 1931, 'CH': 24988}
chunks by language    : {'en': 15136, 'de': 26919}
ECHR chunks by section : {'HEADER': 1508, 'LAW': 6952, 'PROCEDURE': 476, 'FACTS': 4567, 'OPERATIVE': 580, 'unparsed': 1053}


### 4b. Inspect genre — distribution + cross-check the parser-flag (before wiring it in)
Prints genre per source, then **cross-tabulates** the existing *unparsed* parser-flag
(`echr_sections() is None`) against the marker-based `communicated` detector, and reports any
disagreement. If the flag and the markers diverge badly the genre is unreliable and the run
stops here for inspection rather than silently filtering.

In [ ]:
GENRE_RELIABLE = True   # set False by the cross-check below if flag vs markers disagree badly

if records:
    # genre per source — at the document level and the chunk level
    print("GENRE per source (documents):")
    for src in ("echr", "ris", "swiss"):
        srecs = [r for r in records if r["source"] == src]
        if srecs:
            print(f"  {src:5s}: {dict(Counter(assign_genre(r) for r in srecs))}")
    print("GENRE per source (chunks):")
    for src in ("echr", "ris", "swiss"):
        d = dict(Counter(c['genre'] for c in chunks if c['source'] == src))
        if d:
            print(f"  {src:5s}: {d}")

    echr_recs = [r for r in records if r["source"] == "echr"]

    # --- cross-tab: existing unparsed parser-flag  ×  marker-based communicated detector ---
    print("\nCross-check (ECHR docs): unparsed-flag  ×  marker detector")
    ct = Counter()
    disagree_unparsed_not_comm, disagree_comm_parsed = [], []
    for r in echr_recs:
        unparsed = echr_sections(r["text"]) is None              # the existing parser flag
        comm_marker = echr_genre_from_markers(r["text"]) == "communicated"
        ct[(unparsed, comm_marker)] += 1
        if unparsed and not comm_marker:
            disagree_unparsed_not_comm.append(r)                 # unparsed but a real judgment
        if comm_marker and not unparsed:
            disagree_comm_parsed.append(r)                       # 'communicated' yet parsed
    print(f"  unparsed=True  & communicated-marker=True : {ct[(True, True)]:4d}   (agree: pending)")
    print(f"  unparsed=False & communicated-marker=False: {ct[(False, False)]:4d}   (agree: decided)")
    print(f"  unparsed=True  & communicated-marker=False: {ct[(True, False)]:4d}   <- judgments the parser missed")
    print(f"  unparsed=False & communicated-marker=True : {ct[(False, True)]:4d}   <- 'communicated' yet parsed")

    n_dis = len(disagree_unparsed_not_comm) + len(disagree_comm_parsed)
    n_comm = sum(1 for r in echr_recs if echr_genre_from_markers(r["text"]) == "communicated")
    print(f"\n  communicated (markers): {n_comm} | disagreements with parser-flag: {n_dis}")
    for r in disagree_unparsed_not_comm[:5]:
        print(f"    unparsed-but-not-communicated: {r['title'][:55]} (doctype={r['meta'].get('doctype')})")

    # verdict: the parser-flag is a PROXY; genre must come from doctype+markers, not the flag.
    # Unreliable only if marker-communicated and the explicit doctype=HECOM diverge.
    hecom = sum(1 for r in echr_recs if (r["meta"].get("doctype") or "").upper() == "HECOM")
    n_doctype_vs_marker = sum(
        1 for r in echr_recs
        if ((r["meta"].get("doctype") or "").upper() == "HECOM")
        != (echr_genre_from_markers(r["text"]) == "communicated"))
    print(f"\n  doctype=HECOM: {hecom} | doctype vs marker disagreements: {n_doctype_vs_marker}")
    GENRE_RELIABLE = n_doctype_vs_marker <= max(3, int(0.02 * len(echr_recs)))
    if GENRE_RELIABLE:
        print("  VERDICT: genre is reliable (doctype agrees with markers). "
              "Using doctype as primary; the unparsed parser-flag is a proxy only "
              f"({len(disagree_unparsed_not_comm)} real judgments are unparsed and must NOT be dropped).")
    else:
        print("  !! VERDICT: doctype and markers disagree beyond tolerance — STOP and inspect "
              "before wiring genre into retrieval. (Do not filter on an unreliable signal.)")

    # non-ECHR sanity: German sources must never inherit the ECHR communicated genre
    for src in ("ris", "swiss"):
        srecs = [r for r in records if r["source"] == src]
        if not srecs:
            continue
        s_comm = sum(1 for r in srecs if echr_genre_from_markers(r["text"]) == "communicated")
        print(f"\n{src.upper()} sanity: {len(srecs)} records, genres "
              f"{dict(Counter(assign_genre(r) for r in srecs))}; "
              f"ECHR communicated-markers present = {s_comm} (expected 0 — fix is ECHR-only).")
else:
    print("No records loaded — genre report skipped.")

GENRE per source (documents):
  echr : {'admissibility': 311, 'merits': 567, 'communicated': 238}
  ris  : {'principle': 38, 'decision': 479}
  swiss: {'decision': 2007}
GENRE per source (chunks):
  echr : {'admissibility': 2769, 'merits': 11418, 'communicated': 949}
  ris  : {'principle': 38, 'decision': 1893}
  swiss: {'decision': 24988}

Cross-check (ECHR docs): unparsed-flag  ×  marker detector
  unparsed=True  & communicated-marker=True :  225   (agree: pending)
  unparsed=False & communicated-marker=False:  873   (agree: decided)
  unparsed=True  & communicated-marker=False:   18   <- judgments the parser missed
  unparsed=False & communicated-marker=True :    0   <- 'communicated' yet parsed

  communicated (markers): 225 | disagreements with parser-flag: 18
    unparsed-but-not-communicated: I.S. v. GERMANY (doctype=HECOM)
    unparsed-but-not-communicated: CARSTOIU v. ROMANIA (doctype=HECOM)
    unparsed-but-not-communicated: LOPEZ GUIO v. SLOVAKIA (doctype=HECOM)
    unparsed

## 5. Embed + FAISS index (multilingual-e5-base, CPU, exact cosine)
Cached to disk; numpy exact-cosine fallback if `faiss` isn't installed.

In [ ]:
import numpy as np

_embedder = None


def _get_embedder():
    global _embedder
    if _embedder is None:
        from sentence_transformers import SentenceTransformer
        _embedder = SentenceTransformer(EMB_MODEL, device="cpu")
    return _embedder


def embed_passages(texts, progress=True):
    m = _get_embedder()
    return m.encode([f"passage: {t}" for t in texts], batch_size=16, convert_to_numpy=True,
                    normalize_embeddings=True, show_progress_bar=progress).astype("float32")


def embed_query(q):
    m = _get_embedder()
    return m.encode([f"query: {q}"], convert_to_numpy=True,
                    normalize_embeddings=True).astype("float32")


class _NumpyExactIndex:
    # exact cosine via brute-force dot product (identical to faiss.IndexFlatIP)
    def __init__(self, mat):
        self._m = mat
        self.ntotal = int(mat.shape[0])

    def search(self, q, k):
        sims = (q @ self._m.T)[0]
        k = min(k, self._m.shape[0])
        top = np.argpartition(-sims, k - 1)[:k]
        top = top[np.argsort(-sims[top])]
        return sims[top][None, :], top[None, :]


EMBED_CHECKPOINT_EVERY = 2000    # chunks per cache save (a CPU run of hours must survive a kill)


def _load_cached_rows():
    """Incremental cache: {chunk_id: row}. Migrates the legacy whole-corpus cache
    (keys 'key'+'emb', ECHR + RIS-principle chunks in build order) on first load."""
    if not EMB_CACHE.exists():
        return {}
    cached = np.load(EMB_CACHE, allow_pickle=True)
    if "ids" in cached.files:                     # current per-chunk format
        ids = [str(x) for x in cached["ids"]]
        return dict(zip(ids, cached["emb"].astype("float32")))
    # legacy format: reconstruct the old id order = ECHR chunks + RIS principle chunks,
    # in current corpus order (ECHR chunking and RIS principle ids are unchanged).
    legacy_ids = [c["chunk_id"] for c in chunks
                  if c["source"] == "echr" or (c["source"] == "ris" and c["genre"] == "principle")]
    emb = cached["emb"].astype("float32")
    if len(legacy_ids) == emb.shape[0]:
        print(f"migrated legacy cache: {emb.shape[0]} vectors reused")
        return dict(zip(legacy_ids, emb))
    print(f"legacy cache mismatch ({emb.shape[0]} vectors vs {len(legacy_ids)} expected) — ignoring it")
    return {}


def _save_cache(cache):
    ids = list(cache.keys())
    np.savez(EMB_CACHE, ids=np.array(ids), emb=np.stack([cache[i] for i in ids]))


def build_index(chunk_list):
    if not chunk_list:
        return None
    cache = _load_cached_rows()
    ids = [c["chunk_id"] for c in chunk_list]
    missing = [i for i, cid in enumerate(ids) if cid not in cache]
    print(f"cache: {len(ids) - len(missing)}/{len(ids)} chunks already embedded | to embed: {len(missing)}")
    if missing:
        for b0 in range(0, len(missing), EMBED_CHECKPOINT_EVERY):
            batch = missing[b0:b0 + EMBED_CHECKPOINT_EVERY]
            new_emb = embed_passages([chunk_list[i]["text"] for i in batch], progress=False)
            for i, row in zip(batch, new_emb):
                cache[ids[i]] = row
            _save_cache(cache)
            print(f"  embedded {min(b0 + len(batch), len(missing))}/{len(missing)} new chunks "
                  f"(checkpointed -> {EMB_CACHE.name})")
    emb = np.stack([cache[cid] for cid in ids]).astype("float32")
    try:
        import faiss
        idx = faiss.IndexFlatIP(emb.shape[1])
        idx.add(emb)
        print("index backend: faiss IndexFlatIP (exact)")
        return idx
    except ImportError:
        print("faiss not installed -> numpy exact-cosine fallback (identical results)")
        return _NumpyExactIndex(emb)


def index_matrix(idx):
    # The same normalised e5 vectors already in the flat index, kept in memory for MMR
    # (no re-embed). faiss -> reconstruct_n; numpy fallback -> the stored matrix.
    if idx is None:
        return None
    if hasattr(idx, "_m"):                       # numpy fallback
        return idx._m
    return idx.reconstruct_n(0, idx.ntotal)      # faiss IndexFlatIP


index = None
EMB_MATRIX = None
if chunks:
    try:
        index = build_index(chunks)
        EMB_MATRIX = index_matrix(index)
        print(f"index ready: {index.ntotal} vectors | MMR matrix: {EMB_MATRIX.shape}")
    except ImportError as e:
        print(f"embedding library missing ({e}) -> pip install sentence-transformers")
else:
    print("no chunks — skipping index")

cache: 30295/42055 chunks already embedded | to embed: 11760
/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
  embedded 2000/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
  embedded 4000/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
  embedded 6000/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
  embedded 8000/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
  embedded 10000/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
  embedded 11760/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
faiss not installed -> numpy exact-cosine fallback (identical results)
index ready: 42055 vectors | MMR matrix: (42055, 768)


## 6. Retrieval — genre-aware filtering + MMR diversity
Three stages, all on the **existing** index/vectors (no re-embed, no rebuild):
1. **Fetch wide** — search `fetch_k` (~30) candidates from the flat index.
2. **Genre filter** — drop `genre_filter` genres (default `communicated`) from the candidate
   set *after* the search. Override to `None`/`set()` to include them (e.g. litigation-volume
   questions).
3. **MMR re-rank** — greedily pick `k` (~6): each next pick maximises
   `λ·sim(query) − (1−λ)·max sim(already-picked)`, reusing the in-memory e5 vectors
   (`EMB_MATRIX`). Kills the near-duplicate boilerplate clusters. A cruder hard-dedup
   (`cos > MMR_DUP_CEIL`) is available as a fallback.

Each hit keeps `score, id, jurisdiction, country, date, section, genre, url, title, snippet`
and the full `text` (LLM context).

In [ ]:
def _to_hit(i, score):
    c = chunks[i]
    return {
        "score": float(score), "id": c["doc_id"], "chunk_id": c["chunk_id"],
        "jurisdiction": c["jurisdiction"], "source": c["source"],
        "section": c.get("section", ""), "genre": c.get("genre", ""),
        "country": c["country"], "date": c["date"],
        "url": c["url"], "title": c["title"],
        "snippet": sentence_snippet(c["text"]),   # sentence-aligned display
        "text": c["text"],                        # full chunk for generation
    }


def search_candidates(query, fetch_k=FETCH_K):
    # one FAISS/numpy search; returns the query vector + [(chunk_idx, cosine), ...] desc
    if index is None:
        return None, []
    qv = embed_query(query)
    scores, idxs = index.search(qv, fetch_k)
    cand = [(int(i), float(s)) for s, i in zip(scores[0], idxs[0]) if i >= 0]
    return qv, cand


def mmr_rerank(qv, cand, k=TOP_K, mmr_lambda=MMR_LAMBDA, dup_ceil=MMR_DUP_CEIL):
    # Maximal Marginal Relevance over the candidate set, reusing EMB_MATRIX (no re-embed).
    # next pick = argmax  λ·sim(query) − (1−λ)·max sim(already-picked).
    if not cand:
        return []
    if EMB_MATRIX is None:                         # degrade: plain relevance order
        return cand[:k]
    rel = {i: s for i, s in cand}
    pool = [i for i, _ in cand]
    selected = []
    while pool and len(selected) < k:
        if not selected:
            best = max(pool, key=lambda i: rel[i])
        else:
            sel_mat = EMB_MATRIX[selected]
            def _mmr_score(i):
                div = float(np.max(EMB_MATRIX[i] @ sel_mat.T))   # max sim to picked
                # cruder fallback (toggle): drop near-duplicates outright
                # if div > dup_ceil: return -1e9
                return mmr_lambda * rel[i] - (1.0 - mmr_lambda) * div
            best = max(pool, key=_mmr_score)
        selected.append(best)
        pool.remove(best)
    return [(i, rel[i]) for i in selected]


def retrieve(query, k=TOP_K, fetch_k=FETCH_K, genre_filter=LAW_GENRE_EXCLUDE,
             use_mmr=True, mmr_lambda=MMR_LAMBDA):
    # genre_filter: iterable of genres to exclude (default {'communicated'});
    #               None/empty set = keep all genres.
    qv, cand = search_candidates(query, fetch_k=max(fetch_k, k))
    if not cand:
        return []
    if genre_filter:
        gf = set(genre_filter)
        cand = [(i, s) for i, s in cand if chunks[i].get("genre") not in gf]
    picked = mmr_rerank(qv, cand, k=k, mmr_lambda=mmr_lambda) if use_mmr else cand[:k]
    return [_to_hit(i, s) for i, s in picked]


print("retrieve() ready — genre filter + MMR")

retrieve() ready — genre filter + MMR


## 7. Abstention — Layer 1 (out-of-paradigm), deterministic
Catches aggregate/counting/statistical questions before generation; returns an abstention
plus the relevant cases. Explicit `abstained`/`abstain_type` fields. Biased to over-abstain.

In [ ]:
AGGREGATE_PATTERNS = [
    r"\bhow many\b", r"\bhow much\b", r"\bhow often\b",
    r"\bwhat (?:proportion|percentage|share|fraction|number)\b",
    r"\bmost (?:cases|courts|judgments|decisions|of)\b", r"\bmajority of\b",
    r"\bon average\b", r"\baverage\b", r"\bpercentage\b", r"\bproportion\b",
    r"\bnumber of\b", r"\bcount of\b", r"\bfrequenc", r"\brate of\b",
    r"\btrend\b", r"\bover the years\b", r"\bstatistic", r"\btypically\b",
    r"\busually\b", r"\bin general\b",
    r"\bwie viele\b", r"\bwie häufig\b", r"\bwie oft\b", r"\banteil\b",
    r"\bdurchschnitt", r"\bprozent\b", r"\bmehrheit\b", r"\bdie meisten\b",
    r"\bin der regel\b", r"\bstatistik", r"\bhäufigkeit\b", r"\btendenz\b",
    r"\bquote\b", r"\bzahl der\b", r"\binsgesamt\b", r"\bgesamtzahl\b",
]
_AGG_RE = re.compile("|".join(AGGREGATE_PATTERNS), re.IGNORECASE)


def aggregate_trigger(query):
    m = _AGG_RE.search(query or "")
    return m.group(0) if m else None


ABSTAIN_MSG_AGGREGATE = (
    "Abstaining: this is an aggregate/statistical question (counts, proportions, averages, "
    "trends). This tool reads a handful of individual passages and cannot compute corpus-wide "
    "statistics without fabricating them. The most relevant individual cases are listed below.")
NOT_FOUND = "Not found in the corpus"

print(f"Type-1 detector ready ({len(AGGREGATE_PATTERNS)} patterns)")

Type-1 detector ready (36 patterns)


## 8. Grounded generation (Ollama optional) + Layer-2 abstention
Layer 2 enforced in the prompt (reply exactly "Not found in the corpus"). Degrades to
retrieval-only if Ollama is absent. Context uses each hit's full chunk text + section tag.

In [ ]:
OLLAMA_OK = False
if USE_LLM:
    try:
        import ollama
        OLLAMA_OK = True
    except Exception as e:
        print(f"Ollama unavailable ({e}) — retrieval-only mode")

SYSTEM_PROMPT = (
    "You are a legal retrieval assistant over a multi-jurisdiction corpus "
    "(ECHR, Austrian OGH, Swiss courts). Answer ONLY from the numbered SOURCES provided.\n"
    "1. Use no outside knowledge. If the sources do not support an answer, reply "
    "EXACTLY: Not found in the corpus\n"
    "2. For every statement, name the jurisdiction (ECHR, AT (OGH) or CH) and cite the source "
    "id in square brackets, e.g. [echr:001-157293:2].\n"
    "3. Never attribute one jurisdiction's rule to another; never merge jurisdictions.\n"
    "4. Be concise and factual.")


def build_context(hits):
    blocks = []
    for n, h in enumerate(hits, 1):
        sec = f" | section={h['section']}" if h.get("section") else ""
        blocks.append(f"[{n}] id={h['chunk_id']} | jurisdiction={h['jurisdiction']}{sec} | "
                      f"title={h['title']}\n{h['text']}")        # full chunk text
    return "\n\n".join(blocks)


def generate(query, hits):
    if not OLLAMA_OK or not hits:
        return None
    user = (f"SOURCES:\n{build_context(hits)}\n\nQUESTION: {query}\n\n"
            "Answer using only the sources above, with jurisdiction + [id] citations.")
    try:
        resp = ollama.chat(model=GEN_MODEL, messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}])
        return resp["message"]["content"].strip()
    except Exception as e:
        return f"[generation unavailable: {e}]"


# Layer-3 (genre) abstention message — surfaced alongside the communicated cases as context.
ABSTAIN_MSG_BOILERPLATE = (
    "Abstaining: pending applications on this topic exist in the corpus (ECHR communicated "
    "cases — listed separately below as CONTEXT), but no decided ruling states the principle, "
    "so the law cannot be provided. Communicated cases restate the dispute and pose questions; "
    "they do not hold.")


def _communicated_context(query, genre_filter, k):
    # Pull the genres we just excluded (the boilerplate) so they can be shown as context,
    # never as the answer basis. Same vectors, separate MMR pass.
    if not genre_filter:
        return []
    qv, cand = search_candidates(query, fetch_k=FETCH_K)
    keep = set(genre_filter)
    comm = [(i, s) for i, s in cand if chunks[i].get("genre") in keep]
    return [_to_hit(i, s) for i, s in mmr_rerank(qv, comm, k=k)]


def answer(query, k=TOP_K, genre_filter=LAW_GENRE_EXCLUDE):
    result = {"query": query, "abstained": False, "abstain_type": None,
              "abstain_reason": None, "hits": [], "context": [], "answer": None,
              "mode": "generate" if OLLAMA_OK else "retrieval_only"}
    # substantive retrieval: communicated routed out, MMR-diversified
    hits = retrieve(query, k=k, genre_filter=genre_filter)
    result["hits"] = hits
    # the excluded boilerplate, surfaced separately as inspectable context
    result["context"] = _communicated_context(query, genre_filter, k)

    # Layer 1 — out-of-paradigm aggregate question
    trig = aggregate_trigger(query)
    if trig:
        result.update(abstained=True, abstain_type="type1_out_of_paradigm",
                      abstain_reason=f"aggregate pattern: '{trig}'", answer=ABSTAIN_MSG_AGGREGATE)
        return result
    # if hits and hits[0]["score"] < SCORE_FLOOR:
    #     result.update(abstained=True, abstain_type="type2_low_score", answer=NOT_FOUND); return result

    # Layer 3 — genre/boilerplate: nothing substantive survived, only communicated boilerplate.
    if not hits:
        if result["context"]:
            result.update(abstained=True, abstain_type="type3_boilerplate_only",
                          abstain_reason="boilerplate_only", answer=ABSTAIN_MSG_BOILERPLATE)
        else:
            result.update(abstained=True, abstain_type="type2_no_hits", answer=NOT_FOUND)
        return result

    result["answer"] = generate(query, hits)
    return result


print(f"answer() ready — mode: {'generate' if OLLAMA_OK else 'retrieval_only'} | "
      "abstention layers: 1 aggregate · 2 no-hits · 3 boilerplate_only")

answer() ready — mode: generate | abstention layers: 1 aggregate · 2 no-hits · 3 boilerplate_only


## 9. Display helper
Each line shows **date + jurisdiction + section**; then a sentence-aligned snippet.

In [ ]:
def _show_hit(n, h):
    date = h.get("date") or "n.d."
    print(f"  [{n}] cos={h['score']:.3f} | {h['jurisdiction']:9s} | {date:10s} | "
          f"{(h.get('country') or ''):4s} | {h.get('genre', ''):13s} | "
          f"{h.get('section', ''):10s} | {h['title'][:40]}")
    print(f"      {h['chunk_id']}  {h['url']}")
    print(f"      {h['snippet']}")


def show(result):
    print(f"Q: {result['query']}")
    line = f"mode={result['mode']}  abstained={result['abstained']}"
    if result["abstained"]:
        line += f"  type={result['abstain_type']}  ({result['abstain_reason']})"
    print(line)
    if result["answer"]:
        print(f"\nANSWER:\n{result['answer']}")
    print(f"\nRETRIEVED — substantive ({len(result['hits'])}):")
    for n, h in enumerate(result["hits"], 1):
        _show_hit(n, h)
    ctx = result.get("context") or []
    if ctx:
        print(f"\nCONTEXT — communicated cases routed out of the answer ({len(ctx)}):")
        for n, h in enumerate(ctx, 1):
            _show_hit(n, h)
    print("=" * 80)


print("show() ready")

show() ready


## 10. Demo — per-source queries (guarded)
Queries are phrased in each court's own register, which retrieves better than the abstract
label **"parental alienation"**. Caveat: in this corpus that label is polysemous — it also
matches *cultural/religious estrangement* cases (and the Austrian criminal sense of
*Entfremdung* = misappropriation), so concrete legal phrasings are used instead. The ECHR
hits should now come from the Court's assessment (`section=LAW`/`FACTS`), not quoted statutes.

In [ ]:
DEMO_QUERIES = [
    "Wann ist von einer Vollzugsmaßnahme abzusehen, wenn sie dem Kindeswohl widerspricht?",
    "Voraussetzungen für gemeinsame Obsorge bei fehlender Kommunikationsbasis der Eltern",
    "enforcement of contact rights where one parent obstructs the relationship with the other parent",
    "positive obligations of the State to maintain contact between parent and child",
    "Welche positiven Pflichten hat der Staat, um den Kontakt zwischen Elternteil und Kind aufrechtzuerhalten?",
    "Unter welchen Voraussetzungen kann einem Elternteil die Obhut über das Kind entzogen werden?",   # Swiss register (Obhut)
    "Wie wird das Besuchsrecht geregelt, wenn das Kind den Kontakt zum anderen Elternteil ablehnt?",
    "Wie viele Urteile betreffen die Durchsetzung des Kontaktrechts?",
]

if not chunks:
    print("Demo skipped — input data not found.")
    for _, p in SOURCES:
        print(f"   expected: {p}")
elif index is None:
    print("Demo skipped — index unavailable. pip install sentence-transformers faiss-cpu")
else:
    for q in DEMO_QUERIES:
        show(answer(q))

Q: Wann ist von einer Vollzugsmaßnahme abzusehen, wenn sie dem Kindeswohl widerspricht?
mode=generate  abstained=False

ANSWER:
Die Vollzugsmaßnahmen im Sinne des § 110 Abs 2 AußStrG iVm § 79 Abs 2 AußStrG sind gekennzeichnet durch die Verwirklichung des Leistungsbefehles unter Wahrung der Interessen aller Beteiligten, aber unter Hintansetzung von „schädigender Zweifelsucht und Ängstlichkeit".

Auch für das Absehen von einer Vollzugsmaßnahme ist ausschließlich das Kindeswohl maßgebliches Kriterium. Allerdings reicht ein bloßer Widerwille des Kindes gegen das Besuchsrecht ebensowenig wie ein Widerwille des anderen Elternteils aus, eine Gefährdung des Kindeswohles im Sinne des § 110 Abs 3 AußStrG zu bejahen.

Das Kindeswohl ist gefährdet, wenn die Vollzugsmaßnahme dem Kind selbst persönlichen Verkehr mit dessen Willen entzieht. Eine solche Gefährdung muss jedoch über die zwangsläufigen Folgen eines erneuten Aufenthaltswechsels hinausgehen.

Das Kindeswohl ist gefährdet, wenn eine Vollzug

## 11. Before / after — the ECHR boilerplate fix on the example query
The capability-boundary question is *"can the system tell when it has no law to state?"*. The
old top-k looked confident (cosine ~0.88) while being **all communicated boilerplate** —
high-similarity, zero-answer-value, near-duplicate. The fix routes that genre out of
law-questions and MMR-diversifies what remains, and — when *nothing but* boilerplate matches
— the new `type3_boilerplate_only` abstention says so honestly instead of paraphrasing a
pending application back as if it were settled law. Genre is also the field that makes the
*other* question answerable ("how often is this litigated") by including the same cases.

In [ ]:
EXAMPLE_QUERY = ("enforcement of contact rights where one parent obstructs the "
                 "relationship with the other parent")


def _print_ranking(label, hits):
    print(label)
    if not hits:
        print("    (none)")
    for n, h in enumerate(hits, 1):
        print(f"  [{n}] cos={h['score']:.3f} | {h.get('genre',''):13s} | "
              f"{h.get('section',''):10s} | {(h.get('country') or ''):4s} | {h['title'][:46]}")


if not chunks or index is None:
    print("Before/after skipped — data or index unavailable.")
    for _, p in SOURCES:
        if not p.exists():
            print(f"   missing: {p}")
else:
    # BEFORE: old behaviour — no genre filter, no MMR, plain top-6
    before = retrieve(EXAMPLE_QUERY, k=6, genre_filter=None, use_mmr=False)
    # AFTER: genre filter (drop communicated) + MMR diversity
    after = retrieve(EXAMPLE_QUERY, k=6)

    print(f"Q: {EXAMPLE_QUERY}\n")
    _print_ranking("BEFORE  (no genre filter, no MMR):", before)
    print()
    _print_ranking("AFTER   (exclude communicated + MMR):", after)

    b_comm = sum(1 for h in before if h.get("genre") == "communicated")
    b_docs = len({h["id"] for h in before})
    a_docs = len({h["id"] for h in after})
    print(f"\n  before: {b_comm}/6 communicated boilerplate, {b_docs} distinct documents")
    print(f"  after : {sum(1 for h in after if h.get('genre')=='communicated')}/6 communicated, "
          f"{a_docs} distinct documents")

    # genre breakdown of the ECHR corpus (documents)
    echr_recs = [r for r in records if r["source"] == "echr"]
    print("\nECHR documents by genre:",
          dict(Counter(assign_genre(r) for r in echr_recs)))

Q: enforcement of contact rights where one parent obstructs the relationship with the other parent

BEFORE  (no genre filter, no MMR):
  [1] cos=0.879 | communicated  | unparsed   | ROU  | CAERIDIN v. ROMANIA
  [2] cos=0.879 | communicated  | unparsed   | ROU  | TOIA v. ROMANIA
  [3] cos=0.877 | communicated  | unparsed   | ROU  | IONEL v. ROMANIA
  [4] cos=0.876 | communicated  | unparsed   | ROU  | BUȘ v. ROMANIA
  [5] cos=0.875 | communicated  | unparsed   | GRC  | ANAGNOSTAKIS v. GREECE
  [6] cos=0.874 | communicated  | unparsed   | NOR  | D.R. v. NORWAY

AFTER   (exclude communicated + MMR):
  [1] cos=0.870 | admissibility | HEADER     | DEU  | BUSSMANN v. GERMANY
  [2] cos=0.866 | merits        | FACTS      | POL  | CASE OF MALEC v. POLAND
  [3] cos=0.864 | merits        | LAW        | ROU  | CASE OF CRISTESCU v. ROMANIA
  [4] cos=0.863 | merits        | FACTS      | DEU  | CASE OF BUCHLEITHER v. GERMANY
  [5] cos=0.869 | merits        | LAW        | DEU  | CASE OF HOPPE v. GERMA